In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
import matplotlib.pyplot as plt

In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/NYTimes/code/Initialization/InitializationHourly.ipynb"

In [ ]:
df_lead_paragraph=df_lead_paragraph.sort_values(by="Date")

In [ ]:
df_sentiment.columns

In [ ]:
df_lead_paragraph = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/TimeSentiment.pkl")

In [ ]:
df_sentiment.loc[df_sentiment["Date"]<"2020-06-01"]

In [ ]:
df_sentiment[['Sentiment_ln_return',"Hour"]]#.groupby("Hour").count()

In [ ]:
fig, axs = plt.subplots(4, 3,figsize=(21,24))

i=0;j=0
df_lead_paragraph.groupby("Hour").count()["headline"].plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="2011-01-04 to 2022-03-08",ax=axs[i,j])
j+=1
df_lead_paragraph.loc[df_lead_paragraph["Date"]<"2020-06-01"].groupby("Hour").count()["headline"].plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="2011-01-04 to 2020-06-01",ax=axs[i,j])
j+=1
df_lead_paragraph.loc[df_lead_paragraph["Date"]>"2020-06-01"].groupby("Hour").count()["headline"].plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="2020-06-01 to 2022-03-08",ax=axs[i,j])
j=0
i+=1
df_sentiment[['Sentiment_ln_return',"Hour"]].groupby("Hour").count().plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="2011-01-04 to 2022-03-08",ax=axs[i,j])
j+=1
df_sentiment.loc[df_sentiment["Date"]<"2020-06-01"][['Sentiment_ln_return',"Hour"]].groupby("Hour").count().plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="2011-01-04 to 2020-06-01",ax=axs[i,j])
j+=1
df_sentiment.loc[df_sentiment["Date"]>"2020-06-01"][['Sentiment_ln_return',"Hour"]].groupby("Hour").count().plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="2020-06-01 to 2022-03-08",ax=axs[i,j])

# df_sentiment[['Sentiment',"Hour"]].groupby("Hour").count()
# sns.histplot(df_sentiment['Sentiment'].groupby("Hour").count(), bins=100, kde=True,ax=axs[i,j])
j+=1

# sns.histplot(df['Sentiment'].dropna(), bins=100, kde=True)


# sns.histplot(df['Positive Natural Log'].dropna(), bins=100, kde=True)

# plt.subplot(10, 1, 4)
# sns.histplot(df['Positive'].dropna(), bins=100, kde=True)

# plt.subplot(10, 1, 5)
# sns.histplot(df['Negative Natural Log'].dropna(), bins=100, kde=True)

# plt.subplot(10, 1, 6)
# sns.histplot(df['Negative'].dropna(), bins=100, kde=True)

# plt.subplot(10, 1, 7)
# sns.histplot(df['Neutral Natural Log'].dropna(), bins=100, kde=True)

# plt.subplot(10, 1,8)
# sns.histplot(df['Neutral'].dropna(), bins=100, kde=True)

# plt.subplot(10, 1,9)
# sns.histplot(df['Polarity'].dropna(), bins=100, kde=True)

# plt.subplot(10, 1,10)
# sns.histplot(df['Subjectivity'].dropna(), bins=100, kde=True)


# plt.tight_layout()

# # plt.autofmt_xdate()
# # fig = plt.get_figure()
# # Could have also done:
# #AAPL['Daily Return'].hist()
# plt.suptitle("New York Times\n",fontsize=26,weight='bold')
# plt.subplots_adjust(top=0.97)
# plt.tight_layout()
# plt.savefig("/Users/Farid Srouji/Documents/Research/master/NYTimes/images/Histogram/Sentiment10x1.png", dpi=216,bbox_inches='tight')
# plt.show()
# df_sentiment.groupby("Hour").count()["lead_paragraph"].plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="2011-01-04 to 2022-03-08",ax=axs[i,j])
# j+=1
# df_sentiment.loc[df_sentiment["Date"]<"2020-06-01"].groupby("Hour")["Sentiment"].count().plot(kind="bar",ylabel="Total number of News lead_paragraph (Articles)",title="2011-01-04 to 2020-06-01",ax=axs[i,j])
# j+=1
# df_sentiment.loc[df_sentiment["Date"]>"2020-06-01"].groupby("Hour")["Sentiment"].count().plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="2020-06-01 to 2022-03-08",ax=axs[i,j])
# j=0
# i+=1
plt.suptitle("New York Times Headlines per Hour of Day",fontsize=18,weight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.savefig("/Users/Farid Srouji/Documents/Research/master/NYTimes/images/Histogram/hours.png", bbox_inches='tight')
plt.show()

In [ ]:
df = df_lead_paragraph.groupby("Hour").count()["headline"]

In [ ]:
plt = axs[0].plot(df.index,df,ylabel="Total number of News Headlines (Articles)",title="Headlines per Hour of Day\n2011-01-04 to 2022-03-08")
plt.savefig("/Users/Farid Srouji/Documents/Research/master/NYTimes/images/Histogram/hours.png", bbox_inches='tight')
plt.show()

In [ ]:
df_lead_paragraph.loc[df_lead_paragraph["Date"]>"2020-06-01"].groupby("Hour").count()["headline"].plot(ylabel="Total number of News Headlines (Articles)",title="Headlines per Hour of Day")

In [ ]:
df_lead_paragraph.loc[df_lead_paragraph["Date"]>"2020-06-01"].groupby("Hour").count()["headline"].plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="Headlines per Hour of Day\n2020-06-01 to 2022-03-08")

In [ ]:
df_lead_paragraph.loc[df_lead_paragraph["Date"]<"2020-06-01"].groupby("Hour").count()["headline"].plot(ylabel="Total number of News Headlines (Articles)",title="Headlines per Hour of Day\n2011-01-04 to 2020-06-01")

In [ ]:
df_lead_paragraph.loc[df_lead_paragraph["Date"]<"2020-06-01"].groupby("Hour").count()["headline"].plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="Headlines per Hour of Day\n2011-01-04 to 2020-06-01")

In [ ]:
df_lead_paragraph.groupby("Hour").count()["headline"].plot(kind="bar",ylabel="Total number of News Headlines (Articles)",title="Headlines per Hour of Day\n2011-01-04 to 2020-06-01")

In [ ]:
from yahoo_fin.stock_info import get_data
PresentDate="08/03/2022"
df_vix = get_data("^VIX", start_date="04/01/2011", end_date=PresentDate, index_as_date = True, interval="1d")
df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/vix.pkl")